# SayPay: transformer comparison (XLM-R vs Laya/mmBERT vs our TF-IDF v3)

Runtime → Change runtime type → **T4 GPU**. Then run all cells (~40 min total).

Every model gets the **same training data, dev set and test sets** as the TF-IDF v3 model
(exported by `scripts/export_splits.py`). At the end, download `preds.zip` and put it in
`ml/preds/` in the repo, then run `python scripts/evaluate.py --preds preds/*` locally.

In [ ]:
# 1. Get the code. For a private repo, create a GitHub token (repo read) and use:
#    !git clone https://<TOKEN>@github.com/alenjoby/SayPay.git
!git clone -b claude/trusting-allen-e40phb https://github.com/alenjoby/SayPay.git
%cd SayPay/ml
!pip -q install -r requirements.txt "transformers>=4.48" accelerate

In [ ]:
# 2. Rebuild the data exactly as for v3 (synthetic + Banking77/ArBanking77/MASSIVE), export splits
!python -m datagen.generate
!python scripts/fetch_external.py
!python scripts/export_splits.py

In [ ]:
# 3. XLM-RoBERTa base (270M, 100 languages incl. Arabic + Hindi)
!python scripts/finetune_transformer.py --model xlm-roberta-base --run xlmr --epochs 4 --lr 3e-5

In [ ]:
# 4. mmBERT base: the encoder Laya-multilingual is built on (newer, 1800+ languages)
!python scripts/finetune_transformer.py --model jhu-clsp/mmBERT-base --run mmbert --epochs 4 --lr 3e-5

In [ ]:
# 5. Laya as shipped, zero-shot (no training on our data)
!pip -q install laya
!python scripts/laya_zeroshot.py --run laya_zeroshot

In [ ]:
# 6. Optional: XLM-R on raw text instead of slot-masked text (ablation)
# !python scripts/finetune_transformer.py --model xlm-roberta-base --run xlmr_raw --field text

In [ ]:
# 7. Full comparison right here, then download the predictions
!python scripts/evaluate.py --preds preds/xlmr preds/mmbert preds/laya_zeroshot --out compare
!sed -n '1,40p' reports/compare.md
!zip -qr preds.zip preds reports/compare.md reports/compare.json
from google.colab import files; files.download("preds.zip")